In [2]:
# @title Robust Full Google Landmark Scraper (All Shards + Italy Filter, Faster)
from google.colab import drive
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import re
import shutil
import tarfile
import time
import subprocess
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

# 1) Mount Google Drive
# Stores progress/output safely so long jobs can resume.
drive.mount('/content/drive')

# 2) Configuration
project_folder = '/content/drive/My Drive/Vision_Project_2026'
os.makedirs(project_folder, exist_ok=True)

metadata_dir = '/content'
archive_dir = os.path.join(project_folder, 'archives')
extract_root = '/content/google_landmark_images'
os.makedirs(archive_dir, exist_ok=True)
os.makedirs(extract_root, exist_ok=True)

full_output_csv = os.path.join(project_folder, 'full_benchmark_dataset.csv')
italy_output_csv = os.path.join(project_folder, 'italy_benchmark_dataset.csv')

# Google Landmark train has many shards. 500 is the commonly used full count.
START_SHARD = 0
MAX_SHARDS = 500  # Set lower for testing (for example 2 or 5).
SAVE_INTERVAL = 100

# Speed controls
MAX_WORKERS = 12               # Increase carefully (8-24 is usually reasonable).
URL_BATCH_SIZE = 2000          # Process unique Wikimedia URLs in batches.
BATCH_PAUSE_SECONDS = 0.2      # Small pause between URL batches.
REQUEST_TIMEOUT_SECONDS = 10
REQUEST_RETRIES = 2

# Storage controls for long runs
KEEP_TAR_ARCHIVES = True
DELETE_EXTRACTED_AFTER_SHARD = True

# Approx Italy bounding box
ITALY_LAT_MIN, ITALY_LAT_MAX = 35.4, 47.2
ITALY_LON_MIN, ITALY_LON_MAX = 6.6, 18.8


def is_in_italy(lat, lon):
    return (
        lat is not None
        and lon is not None
        and ITALY_LAT_MIN <= lat <= ITALY_LAT_MAX
        and ITALY_LON_MIN <= lon <= ITALY_LON_MAX
    )


def ensure_metadata():
    """Download metadata once if missing."""
    train_csv = os.path.join(metadata_dir, 'train.csv')
    label_csv = os.path.join(metadata_dir, 'train_label_to_category.csv')

    if not os.path.exists(train_csv):
        print('Downloading train.csv ...')
        subprocess.run(
            ['wget', '-q', '-O', train_csv, 'https://s3.amazonaws.com/google-landmark/metadata/train.csv'],
            check=True,
        )

    if not os.path.exists(label_csv):
        print('Downloading train_label_to_category.csv ...')
        subprocess.run(
            [
                'wget',
                '-q',
                '-O',
                label_csv,
                'https://s3.amazonaws.com/google-landmark/metadata/train_label_to_category.csv',
            ],
            check=True,
        )

    return train_csv, label_csv


def ensure_output_files():
    cols = ['image_id', 'file_path', 'wiki_url', 'lat', 'lon', 'status', 'shard', 'is_italy']

    if not os.path.exists(full_output_csv):
        pd.DataFrame(columns=cols).to_csv(full_output_csv, index=False)

    if not os.path.exists(italy_output_csv):
        pd.DataFrame(columns=cols).to_csv(italy_output_csv, index=False)


def load_processed_ids():
    if not os.path.exists(full_output_csv):
        return set()

    try:
        existing = pd.read_csv(full_output_csv, usecols=['image_id'])
        return set(existing['image_id'].astype(str).unique())
    except Exception:
        return set()


def download_shard(shard_idx):
    """Download shard tar to Drive archive cache if missing."""
    shard_name = f'images_{shard_idx:03d}.tar'
    local_tar = os.path.join(archive_dir, shard_name)

    if os.path.exists(local_tar):
        return local_tar

    url = f'https://s3.amazonaws.com/google-landmark/train/{shard_name}'
    cmd = ['wget', '-q', '-c', '-O', local_tar, url]
    result = subprocess.run(cmd)

    if result.returncode != 0:
        if os.path.exists(local_tar):
            try:
                os.remove(local_tar)
            except OSError:
                pass
        return None

    return local_tar


def get_shard_image_paths(tar_path):
    """Return extracted image paths for a shard, based on tar members."""
    names = []
    with tarfile.open(tar_path, 'r') as tar:
        names = [m.name for m in tar.getmembers() if m.isfile() and m.name.lower().endswith('.jpg')]

    top_dirs = sorted({Path(n).parts[0] for n in names if len(Path(n).parts) > 0})

    if names:
        probe = os.path.join(extract_root, names[0])
        if not os.path.exists(probe):
            print(f'Extracting {os.path.basename(tar_path)} ...')
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(path=extract_root)

    paths = [os.path.join(extract_root, n) for n in names]
    return paths, top_dirs


def cleanup_extracted_shard(top_dirs):
    """Delete extracted shard folders to free Colab disk."""
    for top in top_dirs:
        target = os.path.join(extract_root, top)
        if os.path.isdir(target):
            shutil.rmtree(target, ignore_errors=True)


def get_gps_from_wikimedia(url):
    """Extract monument coordinates from Wikimedia GeoHack link."""
    headers = {'User-Agent': 'UniversityProject/1.0 (contact: student@university.edu)'}

    for attempt in range(REQUEST_RETRIES + 1):
        try:
            response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
            if response.status_code != 200:
                continue

            soup = BeautifulSoup(response.content, 'html.parser')
            geohack_link = soup.find('a', href=lambda href: href and 'geohack.toolforge.org' in href)
            if not geohack_link:
                continue

            match = re.search(r'params=([\d\.]+)_([NS])_([\d\.]+)_([EW])', geohack_link['href'])
            if not match:
                continue

            lat, lat_dir, lon, lon_dir = match.groups()
            lat, lon = float(lat), float(lon)
            if lat_dir == 'S':
                lat = -lat
            if lon_dir == 'W':
                lon = -lon
            return lat, lon
        except Exception:
            if attempt < REQUEST_RETRIES:
                time.sleep(0.25 * (attempt + 1))

    return None


def fetch_gps_for_urls(urls, max_workers):
    """Fetch GPS for unique Wikimedia URLs in parallel."""
    if not urls:
        return {}

    out = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_gps_from_wikimedia, url): url for url in urls}
        for future in tqdm(as_completed(futures), total=len(futures), desc='Fetching unique URLs'):
            url = futures[future]
            try:
                out[url] = future.result()
            except Exception:
                out[url] = None

    return out


def flush_batches(full_batch, italy_batch):
    if full_batch:
        pd.DataFrame(full_batch).to_csv(full_output_csv, mode='a', header=False, index=False)
    if italy_batch:
        pd.DataFrame(italy_batch).to_csv(italy_output_csv, mode='a', header=False, index=False)


print('--- Preparing metadata and mappings ---')
train_csv, label_csv = ensure_metadata()
ensure_output_files()
processed_ids = load_processed_ids()
print(f'Already processed image IDs: {len(processed_ids)}')

df_labels = pd.read_csv(train_csv)
df_categories = pd.read_csv(label_csv)
full_df = pd.merge(df_labels, df_categories, on='landmark_id')

# Fast lookup from image id -> Wikimedia category URL
id_to_wiki = (
    full_df[['id', 'category']]
    .dropna(subset=['id', 'category'])
    .drop_duplicates(subset=['id'])
    .set_index('id')['category']
    .to_dict()
)

print(f'Metadata rows: {len(full_df)}')
print(f'Unique image ids with URLs: {len(id_to_wiki)}')

print('\n--- Starting shard loop ---')
full_batch = []
italy_batch = []
processed_this_run = 0
success_this_run = 0
italy_success_this_run = 0

for shard_idx in range(START_SHARD, MAX_SHARDS):
    print(f'\n[Shard {shard_idx:03d}] Download check...')
    tar_path = download_shard(shard_idx)

    if tar_path is None:
        print(f'Shard {shard_idx:03d} not available or download failed, skipping.')
        continue

    try:
        shard_files, shard_top_dirs = get_shard_image_paths(tar_path)
    except Exception as e:
        print(f'Failed reading/extracting shard {shard_idx:03d}: {e}')
        continue

    if not shard_files:
        print(f'No JPG files found in shard {shard_idx:03d}, skipping.')
        continue

    print(f'Images listed in shard {shard_idx:03d}: {len(shard_files)}')

    remaining_items = []
    for file_path in shard_files:
        img_id = Path(file_path).stem
        if img_id in processed_ids:
            continue
        remaining_items.append(
            {
                'image_id': img_id,
                'file_path': file_path,
                'wiki_url': id_to_wiki.get(img_id),
            }
        )

    print(f'Images left in shard {shard_idx:03d}: {len(remaining_items)}')

    if not remaining_items:
        if DELETE_EXTRACTED_AFTER_SHARD:
            cleanup_extracted_shard(shard_top_dirs)
        continue

    # Major speedup: fetch each unique URL once, in parallel.
    unique_urls = sorted({item['wiki_url'] for item in remaining_items if item['wiki_url']})
    print(f'Unique Wikimedia URLs to fetch in shard {shard_idx:03d}: {len(unique_urls)}')

    url_to_gps = {}
    for start in range(0, len(unique_urls), URL_BATCH_SIZE):
        batch_urls = unique_urls[start : start + URL_BATCH_SIZE]
        batch_map = fetch_gps_for_urls(batch_urls, max_workers=MAX_WORKERS)
        url_to_gps.update(batch_map)
        if BATCH_PAUSE_SECONDS > 0:
            time.sleep(BATCH_PAUSE_SECONDS)

    for i, item in enumerate(tqdm(remaining_items, desc=f'Build rows shard {shard_idx:03d}')):
        img_id = item['image_id']
        file_path = item['file_path']
        wiki_url = item['wiki_url']

        if not wiki_url:
            record = {
                'image_id': img_id,
                'file_path': file_path,
                'wiki_url': None,
                'lat': None,
                'lon': None,
                'status': 'missing_metadata',
                'shard': shard_idx,
                'is_italy': False,
            }
        else:
            gps = url_to_gps.get(wiki_url)
            if gps is None:
                record = {
                    'image_id': img_id,
                    'file_path': file_path,
                    'wiki_url': wiki_url,
                    'lat': None,
                    'lon': None,
                    'status': 'no_gps_found',
                    'shard': shard_idx,
                    'is_italy': False,
                }
            else:
                lat, lon = gps
                in_italy = is_in_italy(lat, lon)
                record = {
                    'image_id': img_id,
                    'file_path': file_path,
                    'wiki_url': wiki_url,
                    'lat': lat,
                    'lon': lon,
                    'status': 'success',
                    'shard': shard_idx,
                    'is_italy': in_italy,
                }
                success_this_run += 1
                if in_italy:
                    italy_success_this_run += 1

        full_batch.append(record)
        if record['status'] == 'success' and record['is_italy']:
            italy_batch.append(record)

        processed_ids.add(img_id)
        processed_this_run += 1

        if (i + 1) % SAVE_INTERVAL == 0:
            flush_batches(full_batch, italy_batch)
            full_batch, italy_batch = [], []

    # Save remaining rows from this shard.
    flush_batches(full_batch, italy_batch)
    full_batch, italy_batch = [], []

    # Free local storage between shards.
    if DELETE_EXTRACTED_AFTER_SHARD:
        cleanup_extracted_shard(shard_top_dirs)

    if not KEEP_TAR_ARCHIVES:
        try:
            os.remove(tar_path)
        except OSError:
            pass

# Final summary
final_full_df = pd.read_csv(full_output_csv)
final_italy_df = pd.read_csv(italy_output_csv)

print('\n=== DONE ===')
print(f'Processed this run: {processed_this_run}')
print(f'Successful GPS this run: {success_this_run}')
print(f'Successful Italy GPS this run: {italy_success_this_run}')
print(f'Total rows in full CSV: {len(final_full_df)}')
print(f'Total Italy rows: {len(final_italy_df)}')

print('\nOutput files:')
print(full_output_csv)
print(italy_output_csv)

Mounted at /content/drive
--- Preparing metadata and mappings ---
Already processed image IDs: 16532
Metadata rows: 4132914
Unique image ids with URLs: 4132914

--- Starting shard loop ---

[Shard 000] Download check...
Extracting images_000.tar ...


/tmp/ipykernel_4072/849753577.py:149: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_root)


Images listed in shard 000: 8266
Images left in shard 000: 0

[Shard 001] Download check...
Extracting images_001.tar ...
Images listed in shard 001: 8266
Images left in shard 001: 0

[Shard 002] Download check...
Extracting images_002.tar ...
Images listed in shard 002: 8266
Images left in shard 002: 8266
Unique Wikimedia URLs to fetch in shard 002: 7498


Fetching unique URLs:   0%|          | 0/2000 [00:00<?, ?it/s]

Fetching unique URLs:   0%|          | 0/2000 [00:00<?, ?it/s]

Fetching unique URLs:   0%|          | 0/2000 [00:00<?, ?it/s]

Fetching unique URLs:   0%|          | 0/1498 [00:00<?, ?it/s]

Build rows shard 002:   0%|          | 0/8266 [00:00<?, ?it/s]


[Shard 003] Download check...


KeyboardInterrupt: 